# Criptografía Cuántica: Procedimiento completo

Se importan las librerías:

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

## Distribución cuántica de claves (BB84)

In [9]:
def bb84(longitud_clave=2000, prob_eve=0.0):
    """
    Es el mismo código del tercer módulo, pero en lugar de un booleano
    para considerar Eve, se considera un caso en el que ella intercepta
    una fracción de los qubits
    """
    backend = AerSimulator()

    alice_bits  = np.random.randint(2, size=longitud_clave)
    alice_bases = np.random.randint(2, size=longitud_clave)
    bob_bases   = np.random.randint(2, size=longitud_clave)
    eve_bases   = np.random.randint(2, size=longitud_clave)

    bob_resultados = []
    circuitos      = []

    for i in range(longitud_clave):
        qc = QuantumCircuit(1, 1)

        if alice_bits[i] == 1:
            qc.x(0)
        elif alice_bits[i] == 0:
            # X|0> = |0>
            pass
        if alice_bases[i] == 1:
            qc.h(0)

        # Etapa de Invasión (Eve: Ataque de Interceptar y Reenviar)
        #Esta es la sección que se cambió para atacar según la probabilidad establecidad
        if np.random.rand() < prob_eve:
            
            if eve_bases[i] == 1:
                qc.h(0)
                
            qc.measure(0, 0)

            if eve_bases[i] == 1:
                qc.h(0)

        if bob_bases[i] == 1:
            qc.h(0)

        qc.measure(0, 0)

        circuitos.append(qc)

    qc_transpiled  = transpile(circuitos, backend)
    job            = backend.run(qc_transpiled, shots=1, memory=True) 
    resultados_job = job.result()
    
    for i in range(longitud_clave):
        resultado = int(resultados_job.get_memory(i)[0])
        bob_resultados.append(resultado)

    bob_resultados = np.array(bob_resultados) 
    coincidencias  = alice_bases == bob_bases  

    clave_alice = alice_bits[coincidencias]
    clave_bob   = bob_resultados[coincidencias]

    errores          = np.sum(clave_alice != clave_bob)
    total_bits_utiles = len(clave_alice)
    qber             = (errores / total_bits_utiles) * 100 if total_bits_utiles > 0 else 0.0

    print("="*60)
    print(f" SIMULACIÓN BB84 | Presencia de Eve: {prob_eve}")
    print("="*60)
    print(f"Longitud de transmisión inicial: {longitud_clave} fotones")
    print(f"Longitud de clave cribada: {total_bits_utiles} bits (~50% esperado)")
    print(f"Discrepancias detectadas: {errores} bits")
    print(f"Tasa de Error Cuántico (QBER): {qber:.2f}%")

    return clave_alice, clave_bob, qber

## Estimación del QBER
Para la generación de la clave, se va a escoger un número elevado para la longitud inicial de la clave, tal que se utilizrá el límite asintótico de BB84.

$$
\nu = \sqrt{\frac{\ln(1/\epsilon_{PE})}{2n}}
$$

$$
n\rightarrow\infty \space, \space \therefore\nu\rightarrow 0
$$



Siendo $\nu$ la incertidumbre. Por lo tanto, el umbral conocido de 11% que compromete la seguridad no toma en consideración la incertidumbre estadística (de lo opuesto se ocuparía ajustar el umbral con una desigualdad como la de Hoeffding, la cual no es vista a términos de este curso)

In [10]:
FRACCION_MUESTRA = 0.20  # Fracción sacrificada públicamente para estimar el QBER
UMBRAL_QBER      = 11.0  # Umbral de BB84 incondicional (%)

def estimar_qber(clave_alice, clave_bob, fraccion=FRACCION_MUESTRA):
    """
    Sacrifica una fracción aleatoria de la clave cribada para estimar el QBER
    a través del canal clásico autenticado. Los bits revelados se descartan:
    no pueden usarse para la clave final (corresponden al leakage de estimación).
    """
    n        = len(clave_alice)
    n_muestra = int(n * fraccion)

    # Selección aleatoria de los índices a revelar públicamente
    indices_muestra  = np.random.choice(n, size=n_muestra, replace=False)
    mascara_muestra  = np.zeros(n, dtype=bool)
    mascara_muestra[indices_muestra] = True

    # Comparación pública a través del canal clásico autenticado
    errores_muestra = np.sum(clave_alice[indices_muestra] != clave_bob[indices_muestra])
    qber_estimado   = (errores_muestra / n_muestra) * 100 if n_muestra > 0 else 0.0

    # Descartamos los bits revelados; no pueden usarse para la clave final
    clave_alice_restante = clave_alice[~mascara_muestra]
    clave_bob_restante   = clave_bob[~mascara_muestra]

    print("="*60)
    print(" ESTIMACIÓN DE PARÁMETROS (QBER)")
    print("="*60)
    print(f"Fracción sacrificada: {fraccion*100:.0f}%  |  Bits de muestra: {n_muestra}")
    print(f"Bits restantes para post-procesamiento: {len(clave_alice_restante)}")
    print(f"Errores en muestra: {errores_muestra}")
    print(f"QBER estimado: {qber_estimado:.2f}%")
    if qber_estimado > UMBRAL_QBER:
        print(f"ADVERTENCIA: QBER > {UMBRAL_QBER}% → Protocolo ABORTADO (Eve tiene demasiada informacion)")
    else:
        print(f"OK: QBER <= {UMBRAL_QBER}% → Protocolo continua")

    return qber_estimado, clave_alice_restante, clave_bob_restante

## Reconciliación de Información (Protocolo Cascade)

In [11]:
def _paridad(bits, indices):
    """XOR de los bits en las posiciones dadas"""
    return int(np.sum(bits[indices]) % 2)

def cascade(clave_alice, clave_bob, qber_calculado, num_rondas=4):
    
    clave_bob = clave_bob.copy()
    n         = len(clave_alice)
    leakage   = [0] 

    if qber_calculado <= 0 or n == 0:
        return clave_bob, 0

    # Tamaño inicial de bloque: k1 = 0.73 / e_worst
    k1 = max(2, int(0.73 / (qber_calculado / 100)))

    historial_rondas = []

    def busqueda_binaria(indices):
        if len(indices) == 1:
            clave_bob[indices[0]] ^= 1  # Corrección
            return indices[0]
        mitad = len(indices) // 2
        izq   = indices[:mitad]
        leakage[0] += 1  # Fuga
        if _paridad(clave_alice, izq) != _paridad(clave_bob, izq):
            return busqueda_binaria(izq)
        else:
            return busqueda_binaria(indices[mitad:])

    def backtrack(bit_corregido):
        for bloques_prev, mapa_prev in historial_rondas:
            if bit_corregido not in mapa_prev:
                continue
            bloque_prev = bloques_prev[mapa_prev[bit_corregido]]
            leakage[0] += 1  
            if _paridad(clave_alice, bloque_prev) != _paridad(clave_bob, bloque_prev):
                nuevo_bit = busqueda_binaria(list(bloque_prev))
                backtrack(nuevo_bit) 

    for ronda in range(num_rondas):
        k_ronda = k1 * (2 ** ronda)       
        perm    = np.random.permutation(n) 

        bloques = []
        mapa    = {} 
        for idx_bloque, inicio in enumerate(range(0, n, k_ronda)):
            bloque = perm[inicio : min(inicio + k_ronda, n)]
            bloques.append(bloque)
            for bit in bloque:
                mapa[int(bit)] = idx_bloque

        for bloque in bloques:
            leakage[0] += 1  
            if _paridad(clave_alice, bloque) != _paridad(clave_bob, bloque):
                bit_corregido = busqueda_binaria(list(bloque))
                backtrack(bit_corregido) 

        historial_rondas.append((bloques, mapa))

    errores_residuales = int(np.sum(clave_alice != clave_bob))

    print("="*60)
    print(" RECONCILIACIÓN DE INFORMACIÓN (Cascade)")
    print("="*60)
    print(f"Rondas: {num_rondas}  |  Tamaño inicial de bloque k1: {k1}")
    print(f"Leakage total (bits revelados al canal clásico): {leakage[0]}")
    print(f"Errores residuales tras Cascade: {errores_residuales}")

    return clave_bob, leakage[0]

## Amplificación de Privacidad (Matrices de Toeplitz)

### Min-Entropía Condicional

Como se mencionó en la sección de QBER, $\epsilon_{PE}\rightarrow 0$, lo que significa que el comportamiento uniformemente aleatorio se mantiene así, asumiendo un $\textbf{hardware ideal}$. Por lo tanto:

$$
\lim_{n\rightarrow\infty}H^{\epsilon_{PE}}_{\text{min}}(X|E)=H_{\text{min}}(X|E)
$$

Esto es en realidad la Ley de los Grandes Números aplicado en un caso específico.

Lo siguiente se sale un poco del curso (respecto a lo que esperaría que hicieran), y se muestra solo con tal de ilustrar una implementación formal. Para el caso específico de BB84, la min entropía condicional bajo el límite asintótico es la siguiente:

$$
H_{min}\approx n(1-h_2(\text{QBER}))
$$
Donde $h_2(x)=-x\log_2(x)-(1-x)\log_2(1-x)$

### Matrices de Toeplitz

Siendo conciso, es una matriz donde cada diagonal tiene el mismo valor tal que $T_{i,j}=t_{i-j}$, por ejemplo:
$$
T = \begin{pmatrix}
t_0 & t_{-1} & t_{-2} & t_{-3} & t_{-4} \\
t_1 & t_0 & t_{-1} & t_{-2} & t_{-3} \\
t_2 & t_1 & t_0 & t_{-1} & t_{-2} \\
t_3 & t_2 & t_1 & t_0 & t_{-1}
\end{pmatrix}
$$
De esta forma, cada matriz tiene $n+l-1$ valores.

La clave final es una multiplicación matricial sobre el módulo 2 de la siguiente forma:
$$
k_{final} = T\cdot k_{cascade}\hspace{0.5cm} (\text{mod }2)
$$

Notar como el camino completo de la llave entonces es $k_{\text{BB}84}\rightarrow k_{\text{cascade}}\rightarrow k_{\text{final}}$.

De esta forma, incluso si Eve conoce $T$, no tiene forma de obtener información, tal que después del proceso conoce aun menos del sistema. Esta es la condición de los extractores fuertes, dado que sigue la forma del esquema $Z=\text{Ext}(X,Y)$, donde $Y$ es conocido por Eve y $Z$ es una indistinguible de una distribución generada aleatoriamente.

In [12]:
def amplificacion_privacidad(clave_alice, clave_bob_corregida, leakage_ec,
                              qber_estimado, eps_pa=1e-10):
    """
    La semilla Toeplitz es pública; Alice y Bob aplican el mismo hash de forma independiente.
    """
    n    = len(clave_alice)
    qber = qber_estimado / 100

    # Entropía binaria h₂(p) = −p·log₂(p) − (1−p)·log₂(1−p)
    h2 = (-qber * np.log2(qber) - (1 - qber) * np.log2(1 - qber)) if 0 < qber < 1 else 0.0

    h_min     = n * (1 - h2)                        # H_min(X|E) asintótico para BB84
    penalidad = 2 * np.log2(1 / eps_pa)             # Penalidad uniforme del LHL
    l         = int(np.floor(h_min - leakage_ec - penalidad)) #Ocupo calcular esto para determinar la cantidad mínima de bits que ocupo para que el mensaje sea seguro
    
    print("="*60)
    print(" AMPLIFICACIÓN DE PRIVACIDAD (LHL + Hash Toeplitz)")
    print("="*60)
    print(f"H_min(X|E) ≈ {h_min:.1f} bits  |  leakage_EC: {leakage_ec} bits  |  penalidad_PA: {penalidad:.1f} bits")
    print(f"Longitud de clave segura extraible (l): {l} bits")

    if l <= 0:
        print("ADVERTENCIA: l <= 0 → Protocolo ABORTADO (informacion de Eve no eliminable)")
        return None, None

    # Matriz de Toeplitz (l × n): definida por n+l-1 bits aleatorios (semilla pública)
    # T[i,j] = semilla[n-1+i-j]  →  clave_final = T @ clave  (mod 2)
    semilla_toeplitz = np.random.randint(2, size=n + l - 1)
    indices_i = np.arange(l).reshape(-1, 1)
    indices_j = np.arange(n).reshape(1, -1)
    T = semilla_toeplitz[(n - 1) + indices_i - indices_j]  # Construcción Toeplitz (l × n)

    clave_alice_final = (T @ clave_alice)         % 2
    clave_bob_final   = (T @ clave_bob_corregida) % 2

    coincidencia = np.array_equal(clave_alice_final, clave_bob_final)
    print(f"Claves de Alice y Bob identicas tras PA: {coincidencia}")
    print(f"Clave final: {l} bits = {l // 8} bytes utiles para QOTP")

    return clave_alice_final, clave_bob_final

## Esquema de cifrado (QOTP o Cifrado de Vernam Cuántico)

En el primer módulo, vimos el One Time Pad Clásico y en la tarea se vio su equivalente cuántico, pero directamente programando las matrices para transformar. Ahora, usaremos Qiskit.

Recordar que se ocupan $2n$ bits para encriptar (1 para transformación con matriz de pauli X, otra para matriz de Pauli Z)

In [13]:
def bytes_a_bits(datos_bytes):
    return "".join(f"{b:08b}" for b in datos_bytes)

def bits_a_bytes(bits_array):
    n_bytes = len(bits_array) // 8
    return bytes(int("".join(str(b) for b in bits_array[i*8:(i+1)*8]), 2) for i in range(n_bytes))

def qotp(mensaje_bits, llave_bits):

    n_bits_msg = len(mensaje_bits)
    
    # QOTP requiere el DOBLE de llave que el QOTP clásico
    if len(llave_bits) < 2 * n_bits_msg:
        n_bits_msg = len(llave_bits) // 2
        mensaje_bits = mensaje_bits[:n_bits_msg]

    backend = AerSimulator()
    circuitos = []

    for i in range(n_bits_msg):
        qc = QuantumCircuit(1, 1)

        # 1. Preparación del estado (Alice codifica el mensaje clásico a cuántico)
        if mensaje_bits[i] == 1:
            qc.x(0)

        # 2. Cifrado QOTP (Alice aplica Pauli X y Z según la llave cuántica BB84)
        k_x = llave_bits[2 * i]
        k_z = llave_bits[2 * i + 1]

        if k_x == 1:
            qc.x(0)
        if k_z == 1:
            qc.z(0)

        # 3. Descifrado QOTP (Bob aplica las compuertas inversas)
        # Pauli Z y X son sus propias inversas matemáticas
        if k_z == 1:
            qc.z(0)
        if k_x == 1:
            qc.x(0)

        # 4. Medición (Bob recupera el mensaje clásico original)
        qc.measure(0, 0)
        circuitos.append(qc)

    # Ejecutar simulación de todo el mensaje cifrado
    qc_transpiled = transpile(circuitos, backend)
    job = backend.run(qc_transpiled, shots=1, memory=True)
    resultados = job.result()

    recuperado = [int(resultados.get_memory(i)[0]) for i in range(n_bits_msg)]
    return recuperado, n_bits_msg

## Main: Ejecución Completa

Mucho de estos son un montón de prints para que se vea bonito, así que no hay mucho que ver en términos de código

In [17]:
print("="*80)
print(" PROTOCOLO QKD: ANÁLISIS DE 3 ESCENARIOS DE ATAQUE")
print("="*80 + "\n")

# Pedimos el mensaje una sola vez al inicio para poder probarlo en los 3 escenarios
mensaje_texto = input("Ingrese el mensaje secreto a transmitir: ")

def ejecutar_escenario(nombre_escenario, prob_eve):
    """
    Envuelve el protocolo entero en una función para que los fallos de 
    seguridad no detengan la ejecución del cuaderno de Jupyter (SystemExit),
    sino que simplemente aborten el escenario actual y pasen al siguiente.
    """
    print("\n\n" + "*"*80)
    print(f" INICIANDO ESCENARIO: {nombre_escenario}")
    print("*"*80 + "\n")

    # 1. BB84
    # Llamamos a bb84 con la probabilidad especificada para Eve
    clave_alice_sifted, clave_bob_sifted, _ = bb84(longitud_clave=12000, prob_eve=prob_eve)
    print()

    # 2. Estimación
    qber_calculado, clave_alice_restante, clave_bob_restante = estimar_qber(
        clave_alice_sifted, clave_bob_sifted
    )
    print()

    # Condición de Aborto 1: QBER Excesivo
    if qber_calculado > UMBRAL_QBER:
        print(f">>> PROTOCOLO ABORTADO: QBER en peor caso ({qber_calculado:.2f}%) supera el umbral de seguridad.")
        print(">>> Razón: Eve ha interceptado demasiados bits. El sistema bloquea la comunicación.\n")
        return # Sale de la función, permitiendo que inicie el siguiente escenario

    # 3. Cascade
    clave_bob_corregida, leakage_ec = cascade(clave_alice_restante, clave_bob_restante, qber_calculado)
    print()

    # 4. Amplificación de Privacidad
    clave_alice_final, clave_bob_final = amplificacion_privacidad(
        clave_alice_restante, clave_bob_corregida, leakage_ec, qber_calculado
    )
    print()

    # Condición de Aborto 2: Falla del Hash
    if clave_alice_final is None:
        print(">>> PROTOCOLO ABORTADO: La Amplificación de Privacidad falló.")
        print(">>> Razón: La penalidad de tamaño finito y el leakage eliminaron por completo la clave.\n")
        return

    # 5. QOTP
    print("="*60)
    print(" CIFRADO QUANTUM ONE-TIME PAD (QOTP)")
    print("="*60)

    m = mensaje_texto.encode('utf-8')
    m_bits = [int(b) for b in bytes_a_bits(m)]
    k_bits = clave_alice_final.tolist()

    m_descifrado_bits, bits_procesados = qotp(m_bits, k_bits)

    if bits_procesados < len(m_bits):
        print(f"\nADVERTENCIA: Mensaje truncado. QOTP requiere {2*len(m_bits)} bits de llave, pero solo hay {len(k_bits)} disponibles.")
        m_bits = m_bits[:bits_procesados]

    m_descifrado_bytes = bits_a_bytes(m_descifrado_bits)
    m_descifrado = m_descifrado_bytes.decode('utf-8', errors='ignore')

    print(f"Mensaje original (bits): {''.join(map(str, m_bits))}")
    print(f"Llave consumida (bits):  {''.join(map(str, k_bits[:2*bits_procesados]))}")
    print("-" * 100)
    print(f"\n Mensaje recuperado:   {m_descifrado}")
    print("\n>>> ESCENARIO COMPLETADO CON ÉXITO: Clave segura y mensaje cifrado.")

# Ejecutamos los 3 escenarios automáticamente
ejecutar_escenario("1. CANAL SEGURO (Eve no está presente - 0% intercepción)", prob_eve=0.0)
ejecutar_escenario("2. ATAQUE TOTAL (Eve intercepta TODO - 100% intercepción)", prob_eve=1.0)
ejecutar_escenario("3. ATAQUE PARCIAL (Eve intercepta el 25% de los fotones)", prob_eve=0.25)

 PROTOCOLO QKD: ANÁLISIS DE 3 ESCENARIOS DE ATAQUE



Ingrese el mensaje secreto a transmitir:  Mi nombre es Jeff




********************************************************************************
 INICIANDO ESCENARIO: 1. CANAL SEGURO (Eve no está presente - 0% intercepción)
********************************************************************************

 SIMULACIÓN BB84 | Presencia de Eve: 0.0
Longitud de transmisión inicial: 12000 fotones
Longitud de clave cribada: 6034 bits (~50% esperado)
Discrepancias detectadas: 0 bits
Tasa de Error Cuántico (QBER): 0.00%

 ESTIMACIÓN DE PARÁMETROS (QBER)
Fracción sacrificada: 20%  |  Bits de muestra: 1206
Bits restantes para post-procesamiento: 4828
Errores en muestra: 0
QBER estimado: 0.00%
OK: QBER <= 11.0% → Protocolo continua


 AMPLIFICACIÓN DE PRIVACIDAD (LHL + Hash Toeplitz)
H_min(X|E) ≈ 4828.0 bits  |  leakage_EC: 0 bits  |  penalidad_PA: 66.4 bits
Longitud de clave segura extraible (l): 4761 bits
Claves de Alice y Bob identicas tras PA: True
Clave final: 4761 bits = 595 bytes utiles para QOTP

 CIFRADO QUANTUM ONE-TIME PAD (QOTP)
Mensaje original 